In [1]:
import os
import sys
# 确保能找到你的包
sys.path.append(os.getcwd())

import polars as pl
import numpy as np
from datetime import datetime, timedelta
from mars import MarsDataProfiler
from mars import set_log_level

# 1. 设置环境
set_log_level("WARNING") 

def get_stress_test_data(rows=20000):
    """
    生成包含典型'脏数据'场景的测试集 (加入 NaN 压力测试)。
    """
    np.random.seed(42)
    
    # --- 时间轴生成 ---
    base_date = datetime(2024, 1, 1)
    date_range = [base_date + timedelta(days=x) for x in range(120)]
    dates = np.random.choice(date_range, size=rows)
    is_drift_month = np.array([d >= datetime(2024, 4, 1) for d in dates])
    
    # --- A. 正常对照组 ---
    f_good = np.random.normal(100, 10, size=rows)

    # --- B. 数据质量 (DQ) 报警测试 ---
    
    # [Case 1 Update] 混合缺失爆炸 (Mixed Missing: NaN + -999)
    # 这是一个强压力测试：
    # 既有 NaN (会导致 mean/std 计算崩溃)，又有 -999 (需要逻辑剔除)
    f_missing = np.random.randn(rows)
    rand_mask = np.random.rand(rows)
    
    # 30% 是 -999
    f_missing[rand_mask < 0.3] = -999 
    # 30% 是 np.nan (注意区间是 0.3 到 0.6)
    f_missing[(rand_mask >= 0.3) & (rand_mask < 0.6)] = np.nan
    
    # [Case 2] 唯一值爆炸
    f_id = [f"uid_{i}" for i in range(rows)]
    
    # [Case 3] 单一值报警
    f_const = np.zeros(rows)
    f_const[np.random.rand(rows) < 0.005] = 1 

    # [Case 6 New] 纯 NaN 测试
    # 用于验证基础的 NaN 识别能力
    f_pure_nan = np.random.normal(50, 5, size=rows)
    f_pure_nan[np.random.rand(rows) < 0.2] = np.nan # 20% 是 NaN

    # --- C. 稳定性 (PSI) 报警测试 ---

    # [Case 4] 数值分布漂移
    f_psi_num = np.random.normal(0, 1, size=rows)
    f_psi_num[is_drift_month] = np.random.normal(5, 1, size=np.sum(is_drift_month))

    # [Case 5] 特殊值占比漂移
    f_psi_special = np.random.normal(100, 10, size=rows)
    special_mask_normal = (~is_drift_month) & (np.random.rand(rows) < 0.01)
    f_psi_special[special_mask_normal] = -1
    special_mask_drift = (is_drift_month) & (np.random.rand(rows) < 0.50)
    f_psi_special[special_mask_drift] = -1

    return pl.DataFrame({
        "trans_date": dates.astype("datetime64[ms]"), 
        "feature_0_good": f_good,
        "feature_1_high_missing_mixed": f_missing, # 包含 NaN 和 -999
        "feature_2_high_cardinality": f_id,
        "feature_3_quasi_constant": f_const,
        "feature_4_psi_drift": f_psi_num,
        "feature_5_special_drift": f_psi_special,
        "feature_6_pure_nan": f_pure_nan # 仅包含 NaN
    })

def run_verification():
    print("🚀 生成测试数据 (含 NaN 和 -999)...")
    df = get_stress_test_data()
    
    # 打印一下数据预览，确认 NaN 存在
    print(f"数据预览 (feature_1 应该包含 NaN 和 -999):\n{df.select(['feature_1_high_missing_mixed']).head(5)}")

    print("\n🧐 初始化 Profiler...")
    # 这里定义 -999 为缺失值，但没有显式定义 NaN (因为 NaN 默认就是缺失值)
    profiler = MarsDataProfiler(
        df, 
        missing_values=[-999, "null"], 
        special_values=[-1],
        exclude_features=["trans_date"]
    )

    print("📊 计算画像报告 (启用 Auto Date Aggregation)...")
    
    # 如果代码没有修复，这里会因为 NaN + custom_missing 组合导致 ShapeError
    try:
        report = profiler.generate_profile(
            profile_by="week", 
            dt_col="trans_date"
        )
        print("✅ 报告生成成功！(ShapeError 已修复)")
    except Exception as e:
        print(f"❌ 报告生成失败: {e}")
        return

    # 获取结果进行验证
    overview, _, stats_tables = report.get_profile_data()
    
    print("\n✅ --- 验证结果 (DQ Metrics Verification) ---")
    
    # 1. 验证混合缺失率 (NaN + -999)
    # 预期: 30%(-999) + 30%(NaN) = 60%
    f1_stats = overview.filter(pl.col("feature") == "feature_1_high_missing_mixed")
    missing_rate_1 = f1_stats["missing_rate"][0]
    
    print(f"[Case 1] Mixed Missing (NaN + -999):")
    print(f"   - Expected: approx 0.60 (60%)")
    print(f"   - Actual:   {missing_rate_1:.4f}")
    
    # 宽松断言 (因为是随机生成，留一点波动空间)
    if 0.58 < missing_rate_1 < 0.62:
        print("   -> 🔴 PASS (NaN 和 -999 都被正确识别为缺失值)")
    else:
        print("   -> ❌ FAIL (缺失率计算错误，可能只识别了其中一种)")

    # 2. 验证纯 NaN 缺失率
    f6_stats = overview.filter(pl.col("feature") == "feature_6_pure_nan")
    missing_rate_6 = f6_stats["missing_rate"][0]
    
    print(f"\n[Case 6] Pure NaN:")
    print(f"   - Expected: approx 0.20 (20%)")
    print(f"   - Actual:   {missing_rate_6:.4f}")
    
    if 0.18 < missing_rate_6 < 0.22:
        print("   -> 🔴 PASS")
    else:
        print("   -> ❌ FAIL")

    return report

if __name__ == "__main__":
    report = run_verification()

🚀 生成测试数据 (含 NaN 和 -999)...
数据预览 (feature_1 应该包含 NaN 和 -999):
shape: (5, 1)
┌──────────────────────────────┐
│ feature_1_high_missing_mixed │
│ ---                          │
│ f64                          │
╞══════════════════════════════╡
│ NaN                          │
│ NaN                          │
│ -999.0                       │
│ -999.0                       │
│ NaN                          │
└──────────────────────────────┘

🧐 初始化 Profiler...
📊 计算画像报告 (启用 Auto Date Aggregation)...
✅ 报告生成成功！(ShapeError 已修复)

✅ --- 验证结果 (DQ Metrics Verification) ---
[Case 1] Mixed Missing (NaN + -999):
   - Expected: approx 0.60 (60%)
   - Actual:   0.5982
   -> 🔴 PASS (NaN 和 -999 都被正确识别为缺失值)

[Case 6] Pure NaN:
   - Expected: approx 0.20 (20%)
   - Actual:   0.1989
   -> 🔴 PASS


In [2]:
report

In [3]:
report.show_overview()

feature,dtype,distribution,missing_rate,zeros_rate,unique_rate,top1_ratio,top1_value,mean,std,min,max,p25,median,p75,skew,kurtosis
feature_0_good,Float64,▂▂▄█▆▃▂▂,0.00%,0.00%,100.00%,0.01%,94.21787901780246,100.03,9.93,60.78,144.79,93.24,100.07,106.74,-0.00,-0.01
feature_1_high_missing_mixed,Float64,▂▂▃▆█▅▂▂,59.82%,0.00%,40.18%,29.94%,NaN,0.00,1.00,-4.47,3.60,-0.68,-0.00,0.68,-0.02,-0.00
feature_3_quasi_constant,Float64,█______▂,0.00%,99.44%,0.01%,99.44%,0.0,0.01,0.07,0.00,1.00,0.00,0.00,0.00,13.25,173.58
feature_4_psi_drift,Float64,▂▃█▄▂▃▃▂,0.00%,0.00%,100.00%,0.01%,5.79011061466992,1.20,2.36,-4.00,8.25,-0.44,0.41,2.19,0.91,-0.33
feature_5_special_drift,Float64,▂▂▅█▆▃▂▂,0.00%,0.00%,87.21%,12.80%,-1.0,99.98,9.89,62.05,145.62,93.18,99.95,106.76,0.01,-0.07
feature_6_pure_nan,Float64,▂▂▄█▇▄▂▂,19.89%,0.00%,80.12%,19.89%,NaN,50.00,5.00,31.47,69.15,46.68,49.98,53.34,0.02,-0.00
feature_2_high_cardinality,String,None,0.00%,0.00%,100.00%,0.01%,uid_0,nan,nan,nan,nan,nan,nan,nan,nan,nan


In [4]:
report.show_trend("missing")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_1_high_missing_mixed,Float64,59.57%,59.72%,60.96%,59.08%,59.11%,60.52%,58.20%,55.99%,58.65%,61.10%,59.09%,60.30%,59.46%,61.27%,58.79%,62.35%,62.29%,64.02%,59.82%
feature_3_quasi_constant,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_4_psi_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_5_special_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_6_pure_nan,Float64,19.77%,19.79%,18.96%,18.16%,17.68%,19.40%,22.77%,21.32%,17.80%,19.54%,21.58%,18.75%,21.27%,21.06%,20.12%,21.16%,18.81%,21.95%,19.89%
feature_2_high_cardinality,String,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


In [5]:
report.show_trend("zeros")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_1_high_missing_mixed,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_3_quasi_constant,Float64,99.37%,99.57%,99.60%,99.58%,99.46%,99.57%,99.32%,99.14%,99.59%,99.47%,99.48%,99.32%,99.58%,99.05%,99.56%,99.14%,99.66%,99.39%,99.44%
feature_4_psi_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_5_special_drift,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_6_pure_nan,Float64,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
feature_2_high_cardinality,String,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


In [6]:
report.show_trend("unique")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%
feature_1_high_missing_mixed,Float64,40.61%,40.45%,39.20%,41.09%,41.07%,39.66%,41.97%,44.18%,41.51%,39.08%,41.08%,39.86%,40.70%,38.91%,41.38%,37.82%,37.89%,37.20%,40.18%
feature_3_quasi_constant,Float64,0.18%,0.17%,0.16%,0.17%,0.18%,0.17%,0.17%,0.17%,0.16%,0.18%,0.17%,0.17%,0.17%,0.17%,0.17%,0.17%,0.17%,1.22%,0.01%
feature_4_psi_drift,Float64,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%
feature_5_special_drift,Float64,99.55%,98.97%,98.64%,98.74%,98.93%,99.31%,98.73%,98.97%,98.85%,99.20%,98.78%,99.75%,99.25%,50.78%,48.73%,48.70%,51.20%,51.22%,87.21%
feature_6_pure_nan,Float64,80.32%,80.29%,81.12%,81.92%,82.41%,80.69%,77.32%,78.77%,82.28%,80.55%,78.50%,81.33%,78.81%,79.03%,79.97%,78.93%,81.27%,78.66%,80.12%
feature_2_high_cardinality,String,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%,100.00%


In [7]:
report.show_trend("top1")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total
feature_0_good,Float64,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.08%,0.09%,0.08%,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.09%,0.09%,0.61%,0.01%
feature_1_high_missing_mixed,Float64,30.69%,30.03%,30.88%,30.29%,32.05%,30.78%,29.23%,28.77%,30.11%,32.89%,29.68%,30.66%,30.23%,31.28%,30.88%,31.26%,31.96%,32.32%,29.94%
feature_3_quasi_constant,Float64,99.37%,99.57%,99.60%,99.58%,99.46%,99.57%,99.32%,99.14%,99.59%,99.47%,99.48%,99.32%,99.58%,99.05%,99.56%,99.14%,99.66%,99.39%,99.44%
feature_4_psi_drift,Float64,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.08%,0.09%,0.08%,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.09%,0.09%,0.61%,0.01%
feature_5_special_drift,Float64,0.54%,1.12%,1.44%,1.34%,1.16%,0.78%,1.36%,1.11%,1.23%,0.88%,1.31%,0.34%,0.84%,49.31%,51.36%,51.38%,48.88%,49.39%,12.80%
feature_6_pure_nan,Float64,19.77%,19.79%,18.96%,18.16%,17.68%,19.40%,22.77%,21.32%,17.80%,19.54%,21.58%,18.75%,21.27%,21.06%,20.12%,21.16%,18.81%,21.95%,19.89%
feature_2_high_cardinality,String,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.08%,0.09%,0.08%,0.09%,0.09%,0.08%,0.08%,0.09%,0.09%,0.09%,0.09%,0.61%,0.01%


In [8]:
report.show_trend("psi")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,0.00,0.01,0.01,0.01,0.01,0.00,0.01,0.01,0.02,0.00,0.01,0.01,0.01,0.02,0.00,0.01,0.02,0.03,0.00,0.01,0.0001,0.0000
feature_1_high_missing_mixed,Float64,0.00,0.04,0.03,0.03,0.03,0.06,0.01,0.01,0.05,0.04,0.03,0.04,0.03,0.04,0.05,0.06,0.04,0.24,0.02,0.05,0.0027,1.1188
feature_2_high_cardinality,String,0.00,13.55,13.47,13.52,13.58,13.55,13.53,13.54,13.50,13.57,13.56,13.53,13.52,13.55,13.56,13.55,13.55,15.51,6.35,12.90,10.5768,0.2522
feature_3_quasi_constant,Float64,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000
feature_4_psi_drift,Float64,0.00,0.02,0.01,0.01,0.02,0.02,0.02,0.02,0.01,0.03,0.02,0.02,0.02,21.65,21.81,21.55,21.78,21.45,2.37,6.03,99.4231,1.6547
feature_5_special_drift,Float64,0.00,0.02,0.02,0.02,0.02,0.02,0.01,0.02,0.01,0.02,0.01,0.02,0.01,0.03,0.01,0.01,0.01,0.09,0.01,0.02,0.0004,1.0695
feature_6_pure_nan,Float64,0.00,0.02,0.03,0.01,0.02,0.02,0.02,0.02,0.02,0.01,0.02,0.04,0.04,0.01,0.02,0.03,0.02,0.06,0.01,0.02,0.0002,0.6354


In [9]:
report.show_trend("mean")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,99.95,100.10,100.31,100.24,99.63,100.05,99.92,100.32,100.71,99.71,99.93,100.05,100.17,99.76,99.76,99.62,100.26,99.44,100.03,100.00,0.0988,0.0031
feature_1_high_missing_mixed,Float64,0.01,-0.04,-0.06,0.06,-0.01,-0.04,-0.00,0.01,0.05,0.07,-0.00,-0.07,-0.03,0.02,0.04,-0.05,0.02,0.06,0.00,0.00,0.0019,14.7381
feature_3_quasi_constant,Float64,0.01,0.00,0.00,0.00,0.01,0.00,0.01,0.01,0.00,0.01,0.01,0.01,0.00,0.01,0.00,0.01,0.00,0.01,0.01,0.01,0.0000,0.3207
feature_4_psi_drift,Float64,-0.01,-0.07,0.08,-0.02,-0.03,-0.00,-0.01,0.02,0.00,0.03,0.07,-0.04,-0.02,5.01,5.02,4.97,5.03,4.94,1.20,1.39,5.2989,1.6593
feature_5_special_drift,Float64,99.98,100.06,99.75,100.26,99.80,99.69,100.28,99.82,100.17,99.65,100.03,99.83,100.21,100.03,100.12,100.51,100.03,98.45,99.98,99.93,0.1862,0.0043
feature_6_pure_nan,Float64,50.04,49.85,50.21,49.97,50.17,49.90,50.07,49.80,49.87,49.91,50.13,50.00,49.77,49.99,50.33,49.86,50.13,50.23,50.00,50.01,0.0265,0.0033
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [10]:
report.show_trend("max")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,132.15,139.42,136.11,129.45,129.14,133.78,128.74,128.22,144.79,129.57,132.88,129.70,132.86,131.37,129.38,136.03,134.29,129.97,144.79,132.66,18.5979,0.0325
feature_1_high_missing_mixed,Float64,2.86,2.94,2.99,3.01,3.00,3.28,2.35,2.90,3.16,3.10,2.96,3.60,3.19,2.48,2.78,2.23,2.64,2.68,3.60,2.90,0.1135,0.1163
feature_3_quasi_constant,Float64,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,0.0000,0.0000
feature_4_psi_drift,Float64,3.07,2.90,3.39,3.02,2.95,3.00,3.20,2.72,3.58,3.05,3.05,3.62,3.39,8.23,8.13,7.89,8.25,7.86,8.25,4.52,5.2055,0.5053
feature_5_special_drift,Float64,139.77,132.34,131.87,145.62,131.80,127.30,134.65,134.12,132.06,126.90,132.91,132.58,128.80,137.32,135.11,126.68,127.00,136.57,145.62,132.97,24.2560,0.0370
feature_6_pure_nan,Float64,68.56,67.81,68.47,66.53,64.05,67.97,66.30,67.60,65.31,65.13,69.15,68.53,64.51,67.19,66.92,64.07,68.83,65.01,69.15,66.77,2.9730,0.0258
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [11]:
report.show_trend("min")

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,61.44,66.79,66.64,63.65,60.78,70.27,67.04,70.46,70.32,66.70,67.87,66.65,70.66,65.05,66.24,69.81,63.45,73.22,60.78,67.06,11.3326,0.0502
feature_1_high_missing_mixed,Float64,-2.89,-2.71,-2.92,-2.43,-4.47,-2.87,-2.93,-3.16,-3.63,-2.51,-3.20,-2.63,-3.00,-3.53,-3.43,-2.98,-3.22,-2.69,-4.47,-3.07,0.2333,0.1575
feature_3_quasi_constant,Float64,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0000,0.0000
feature_4_psi_drift,Float64,-4.00,-3.38,-3.13,-3.19,-3.15,-3.29,-3.40,-3.20,-3.11,-3.10,-3.53,-3.54,-3.56,1.65,1.94,1.96,1.87,2.69,-4.00,-1.86,6.2249,1.3417
feature_5_special_drift,Float64,72.15,67.02,69.09,63.43,68.95,66.47,69.87,65.68,72.00,65.84,67.05,62.05,62.55,70.61,72.24,72.50,67.83,75.76,62.05,68.39,14.2096,0.0551
feature_6_pure_nan,Float64,35.44,36.93,34.25,34.80,34.30,35.51,35.04,36.00,35.91,32.87,31.47,34.94,34.56,34.83,33.20,31.56,35.94,35.25,31.47,34.60,2.2017,0.0429
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [12]:
report.show_trend('skew')

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
feature_0_good,Float64,-0.11,0.07,0.01,-0.07,-0.09,0.00,0.03,0.01,0.05,-0.06,-0.06,-0.03,0.05,-0.09,-0.06,0.14,0.09,0.09,-0.00,-0.00,0.0052,27.5130
feature_1_high_missing_mixed,Float64,0.07,0.08,0.11,0.13,-0.18,-0.03,-0.14,0.07,0.02,0.05,-0.06,0.26,-0.04,-0.11,-0.11,-0.20,-0.22,-0.25,-0.02,-0.03,0.0196,4.5471
feature_3_quasi_constant,Float64,12.46,15.15,15.72,15.36,13.55,15.13,12.01,10.67,15.52,13.62,13.73,12.04,15.36,10.10,15.02,10.62,16.97,12.69,13.25,13.65,4.0757,0.1479
feature_4_psi_drift,Float64,0.01,-0.02,-0.07,0.10,-0.04,-0.00,-0.00,-0.11,0.04,-0.11,-0.09,0.14,0.11,0.06,-0.02,0.04,0.06,0.22,0.91,0.02,0.0077,4.9124
feature_5_special_drift,Float64,0.07,0.05,-0.03,0.11,-0.05,-0.13,0.06,-0.03,0.08,-0.07,0.05,0.05,-0.07,0.08,0.01,0.04,-0.12,0.39,0.01,0.03,0.0134,4.1247
feature_6_pure_nan,Float64,0.03,0.25,-0.02,-0.01,-0.09,0.05,-0.07,0.07,0.12,-0.12,-0.05,0.05,-0.05,0.09,0.08,-0.06,0.04,0.10,0.02,0.02,0.0084,4.0236
feature_2_high_cardinality,String,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,0.00,0.0000,0.0000


In [13]:
report.write_excel("mars_demo_report.xlsx")

In [14]:
data  = report.get_profile_data()
data.overview

feature,dtype,distribution,missing_rate,zeros_rate,unique_rate,top1_ratio,top1_value,mean,std,min,max,p25,median,p75,skew,kurtosis
str,str,str,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""feature_0_good""","""Float64""","""▂▂▄█▆▃▂▂""",0.0,0.0,1.0,0.00005,"""94.21787901780246""",100.030674,9.930353,60.775997,144.790843,93.244315,100.065479,106.743678,-0.004967,-0.012534
"""feature_1_high_missing_mixed""","""Float64""","""▂▂▃▆█▅▂▂""",0.59825,0.0,0.40185,0.29935,"""NaN""",0.000107,0.997207,-4.465604,3.602415,-0.677429,-0.001404,0.676057,-0.018504,-0.002242
"""feature_3_quasi_constant""","""Float64""","""█______▂""",0.0,0.9944,0.0001,0.9944,"""0.0""",0.0056,0.074625,0.0,1.0,0.0,0.0,0.0,13.250549,173.57706
"""feature_4_psi_drift""","""Float64""","""▂▃█▄▂▃▃▂""",0.0,0.0,1.0,0.00005,"""5.79011061466992""",1.197297,2.359379,-3.999332,8.245122,-0.444646,0.408244,2.187975,0.906838,-0.333823
"""feature_5_special_drift""","""Float64""","""▂▂▅█▆▃▂▂""",0.0,0.0,0.8721,0.12795,"""-1.0""",99.98415,9.887569,62.053626,145.621147,93.181888,99.947258,106.759385,0.010241,-0.066538
"""feature_6_pure_nan""","""Float64""","""▂▂▄█▇▄▂▂""",0.1989,0.0,0.80115,0.1989,"""NaN""",50.002221,4.996445,31.47425,69.148911,46.684643,49.979092,53.339451,0.020461,-0.004783
"""feature_2_high_cardinality""","""String""",null,0.0,0.0,1.0,0.00005,"""uid_0""",null,null,null,null,null,null,null,null,null


In [15]:
data.stats_trends["psi"]

feature,dtype,2024-01-01,2024-01-08,2024-01-15,2024-01-22,2024-01-29,2024-02-05,2024-02-12,2024-02-19,2024-02-26,2024-03-04,2024-03-11,2024-03-18,2024-03-25,2024-04-01,2024-04-08,2024-04-15,2024-04-22,2024-04-29,total,group_mean,group_var,group_cv
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""feature_0_good""","""Float64""",0.0,0.005403,0.010755,0.007192,0.010564,0.003489,0.012181,0.012814,0.015567,0.003144,0.012055,0.008703,0.008663,0.017019,0.001748,0.014894,0.015221,0.030465,0.002684,0.010549,0.00005,0.0
"""feature_1_high_missing_mixed""","""Float64""",0.0,0.035173,0.032048,0.030855,0.034295,0.059107,0.013294,0.014778,0.046485,0.039402,0.028347,0.044239,0.025851,0.041617,0.046001,0.056664,0.037287,0.243992,0.018461,0.04608,0.002658,1.118822
"""feature_2_high_cardinality""","""String""",0.0,13.547419,13.473914,13.519226,13.584475,13.549153,13.534506,13.542234,13.499204,13.574638,13.558744,13.528536,13.520069,13.554373,13.564015,13.55089,13.545688,15.511873,6.354285,12.89772,10.576829,0.252153
"""feature_3_quasi_constant""","""Float64""",0.0,0.000778,0.001065,0.000884,0.000159,0.000772,0.000035,0.000687,0.000962,0.000178,0.00021,0.00003,0.000881,0.001333,0.000718,0.00073,0.001763,0.000008,0.000087,0.000622,2.5678e-7,0.0
"""feature_4_psi_drift""","""Float64""",0.0,0.017832,0.011305,0.011425,0.0215,0.018426,0.019305,0.021983,0.012492,0.025823,0.016784,0.020648,0.017549,21.654454,21.813386,21.552049,21.783131,21.449362,2.373367,6.02597,99.423083,1.65469
"""feature_5_special_drift""","""Float64""",0.0,0.016195,0.017369,0.016376,0.015926,0.019624,0.009177,0.016325,0.007971,0.015499,0.010738,0.018948,0.014689,0.025376,0.011485,0.010328,0.011475,0.094316,0.006642,0.018434,0.000389,1.069533
"""feature_6_pure_nan""","""Float64""",0.0,0.018488,0.030463,0.01468,0.019864,0.015956,0.015316,0.017726,0.021452,0.005872,0.018306,0.03992,0.035005,0.011061,0.018865,0.027309,0.02278,0.062909,0.009472,0.021998,0.000195,0.635429


In [16]:
# 伪代码逻辑
def filter_bad_features(df):
    
    # 1. 删掉“太无聊”的列 (准常量)
    # 只要 Top1 占比太高，不管它是啥类型，直接删
    top1_ratio = 0.96
    if top1_ratio > 0.95: 
        return "DROP (Quasi-Constant)"

    # 2. 删掉“太杂乱”的 ID 列 (高基数)
    # 只有当它是字符串类型，且几乎都不重复时，才删
    # (数值型的高Unique是好事，不能删！)
    unique_rate = 1.0
    is_string_type = True
    if unique_rate > 0.95 and is_string_type:
        return "DROP (ID Column)"
        
    return "KEEP"

In [17]:
# import polars as pl
# import pandas as pd
# import numpy as np
# import time
# import psutil
# import os
# import gc
# import sys
# from typing import Tuple, Union

# from mars.analysis.profiler import MarsDataProfiler
# from mars.utils.logger import set_log_level, logger

# # ==========================================
# # ⚙️ Stress Test Configuration
# # ==========================================
# # set_log_level("WARNING")  # Reduce IO interference

# # Standard medium-sized risk dataset configuration (200k rows x 2000 cols)
# N_ROWS: int = 200000
# N_COLS: int = 5000
# N_CATS: int = 50
# N_GROUPS: int = 12

# class Colors:
#     GREEN = '\033[92m'
#     RED = '\033[91m'
#     CYAN = '\033[96m'
#     BOLD = '\033[1m'
#     RESET = '\033[0m'

# def get_memory_usage() -> float:
#     """Get current process memory (MB)"""
#     process = psutil.Process(os.getpid())
#     return process.memory_info().rss / 1024 / 1024

# def generate_data() -> pl.DataFrame:
#     """Fast large-scale test data generation"""
#     print(f"{Colors.CYAN}🚀 [DataGen] Generating {N_ROWS:,} rows x {N_COLS} cols...{Colors.RESET}")
#     start = time.time()
    
#     # 1. Numerical columns (Matrix generation is faster)
#     n_num = N_COLS - N_CATS
#     # Using float32 to save memory for the demo, can be float64
#     data = (np.random.randn(N_ROWS, n_num).astype(np.float32) * 10) + 100
    
#     # Inject missing values (-999)
#     mask = np.random.random(data.shape) < 0.1
#     data[mask] = -999
    
#     data_dict = {f"num_{i}": data[:, i] for i in range(n_num)}
    
#     # 2. Categorical columns
#     cats = ["A", "B", "C", "D", "E", "unknown", None]
#     for i in range(N_CATS):
#         data_dict[f"cat_{i}"] = np.random.choice(cats, size=N_ROWS).tolist()
        
#     # 3. Group column
#     groups = [f"2023{m:02d}" for m in range(1, N_GROUPS + 1)]
#     data_dict["month"] = np.random.choice(groups, size=N_ROWS).tolist()
    
#     df = pl.DataFrame(data_dict)
#     size_mb = df.estimated_size('mb')
#     print(f"✅ Data Ready! Size: {size_mb:.2f} MB | Time: {time.time()-start:.2f}s")
    
#     # Check if memory is sufficient for conversion later
#     if size_mb * 3 > psutil.virtual_memory().available / 1024 / 1024:
#         print(f"{Colors.RED}⚠️ Warning: Dataset might be too large for Pandas conversion on this machine.{Colors.RESET}")
        
#     return df

# def run_benchmark_round(df: Union[pl.DataFrame, pd.DataFrame], backend: str) -> Tuple[float, float, float]:
#     """Execute a single round of stress testing"""
#     print(f"\n🔹 Testing Backend: {Colors.BOLD}{backend}{Colors.RESET}")
#     print("-" * 60)
    
#     # Configuration overrides for benchmark (disable sparklines for pure calc speed)
#     # If you want to test sparkline performance, remove "enable_sparkline": False
#     bench_config = {
#         "enable_sparkline": True,
#         # "stat_metrics": ["mean", "std", "min", "max", "p25", "median", "p75", "skew", "kurtosis"]
#         } 

#     # 1. Initialization
#     gc.collect()
#     t0 = time.time()
#     # Initialize Engine
#     profiler = MarsDataProfiler(df, custom_missing_values=[-999, "unknown"])
#     t_init = time.time() - t0
#     print(f"   1. Init Engine       : {t_init:.4f} s")
    
#     # 2. Overview Only (Simulates old get_report)
#     # Using profile_by=None to get global stats
#     t1 = time.time()
#     _ = profiler.generate_profile(profile_by=None, config_overrides=bench_config)
#     t_report = time.time() - t1
#     print(f"   2. Full Overview     : {t_report:.4f} s")
    
#     # 3. Group Profile (Stability Analysis)
#     # Using profile_by="month"
#     t2 = time.time()
#     _ = profiler.generate_profile(
#         profile_by="month", 
#         config_overrides=bench_config # Reuse config to keep sparklines off
#     )
#     t_profile = time.time() - t2
#     print(f"   3. Group Profile (by): {t_profile:.4f} s")
    
#     return t_init, t_report, t_profile

# def print_final_report(pl_times, pd_times):
#     """Print comparison report"""
#     stages = ["Initialization", "Get Full Overview", "Generate Group Profile"]
    
#     print(f"\n{Colors.BOLD}{'🏆 BENCHMARK RESULTS (Time in Seconds)':^65}{Colors.RESET}")
#     print("=" * 65)
#     print(f"| {'Stage':<24} | {'Polars':<10} | {'Pandas':<10} | {'Speedup':<10} |")
#     print("|" + "-"*26 + "+" + "-"*12 + "+" + "-"*12 + "+" + "-"*12 + "|")
    
#     for stage, t_pl, t_pd in zip(stages, pl_times, pd_times):
#         # Calculate speedup
#         speedup = t_pd / t_pl if t_pl > 0 else 0
        
#         # Color code the winner
#         if t_pl < t_pd:
#             pl_str = f"{Colors.GREEN}{t_pl:.4f}{Colors.RESET}"
#             pd_str = f"{t_pd:.4f}"
#         else:
#             pl_str = f"{t_pl:.4f}"
#             pd_str = f"{Colors.GREEN}{t_pd:.4f}{Colors.RESET}"
            
#         print(f"| {stage:<24} | {pl_str:<19} | {pd_str:<19} | {speedup:>9.1f}x |")
#     print("=" * 65)
#     print(f"{Colors.CYAN}* Speedup > 1.0x means Polars is faster.{Colors.RESET}")
#     print(f"{Colors.CYAN}* Note: Sparklines were disabled for pure calculation benchmark.{Colors.RESET}\n")

# # 1. Generate Data (Polars Native)
# df_pl = generate_data()

# # 2. Run Polars Benchmark
# pl_results = run_benchmark_round(df_pl, "Polars (Native)")

# # 3. Convert to Pandas for Comparison
# print(f"\n{Colors.CYAN}🔄 Converting to Pandas for compatibility test...{Colors.RESET}")
# try:
#     t_conv = time.time()
#     df_pd = df_pl.to_pandas()
#     print(f"   Conversion time: {time.time() - t_conv:.2f}s")
    
#     # Explicitly delete Polars DF to free memory if tight
#     # del df_pl 
#     gc.collect()
# except MemoryError:
#     print(f"{Colors.RED}❌ OOM during conversion! Skipping Pandas test.{Colors.RESET}")
#     sys.exit(1)
    
# # 4. Run Pandas Benchmark
# pd_results = run_benchmark_round(df_pd, "Pandas (Compat)")  

# # 5. Show Summary
# print_final_report(pl_results, pd_results)